# Kaleido — Pipeline 2 (v3): Tier Selection & Sequencing

**Job:** Query Supabase P1 tables → run greedy algorithm → compute similarity indices → write 3 Supabase tables + enriched snapshot for P3.

**What this produces:**

| Output | Format | Purpose |
|--------|--------|---------|
| `tier_unit_sequence` | Supabase table | Which units belong to each tier, in order |
| `qbank_unlocks` | Supabase table | Which Q Bank questions unlock at which steps |
| `direction_ref` | Supabase table | Direction ID → human-readable argument text |
| `p2_enriched_snapshot.json` | Google Drive file | Enriched essay data for Pipeline 3 |

**Run cells top to bottom. Only Cell 2 requires your credentials.**

---
## Cell 1 — Install dependencies

In [1]:
!pip install supabase numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


---
## Cell 2 — Your Supabase credentials
Find these in Supabase project → Settings → API

In [2]:
SUPABASE_URL   = "https://upqgkokiemebdxhfdada.supabase.co"
SUPABASE_KEY   = "sb_publishable_YFvR_PWzN_seHZxg9SoJDw_73rhSoM3"

---
## Cell 3 — Connect to Supabase

In [3]:
from supabase import create_client

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ Supabase connected')

✅ Supabase connected


---
## Cell 4 — Generate batch_id

**Important change from v2:** `batch_id` is now a **single new UUID** generated per P2→P3 run.
It represents the complete curriculum version that gets locked to a student at onboarding.
This is NOT the same as P1's internal batch_id (which may be multiple UUIDs from multiple P1 runs).

In [4]:
import uuid
import datetime

# Generate a single UUID for this pipeline run
# This becomes the School's content version identifier
BATCH_ID = str(uuid.uuid4())
GENERATED_AT = datetime.datetime.utcnow().isoformat() + 'Z'

print(f'batch_id     : {BATCH_ID}')
print(f'generated_at : {GENERATED_AT}')
print()
print('This batch_id will be written to all output tables and the P3 snapshot.')
print('Students who onboard after this run will be locked to this batch.')

batch_id     : 00b60e61-22d9-4513-8d08-a8f7c09e6c79
generated_at : 2026-03-12T03:00:58.303884Z

This batch_id will be written to all output tables and the P3 snapshot.
Students who onboard after this run will be locked to this batch.


/tmp/ipykernel_251/4227994227.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  GENERATED_AT = datetime.datetime.utcnow().isoformat() + 'Z'


---
## Cell 5 — Bulk fetch all essay data from P1 tables

Fetches 6 tables from Supabase (all produced by Pipeline 1):
1. `essays` — essay_id, question, structure_type
2. `essay_sentences` — links essays to sentences (with paragraph_type + order)
3. `sentences` — canonical_text, rhetoric_tag, direction_tag, embedding
4. `lexical_items` — phrases within sentences (with embeddings for similarity)
5. `sentence_syntax` — junction table linking sentences to syntax patterns
6. `syntax_items` — the actual syntax pattern strings

In [5]:
def fetch_in_chunks(table, id_col, ids, select_cols, chunk_size=100):
    """Fetch rows by ID list, chunked to avoid Supabase IN-clause limits."""
    results = []
    ids = list(ids)
    for i in range(0, len(ids), chunk_size):
        chunk = ids[i:i + chunk_size]
        resp = supabase.table(table).select(select_cols).in_(id_col, chunk).execute()
        results.extend(resp.data or [])
    return results


print('Fetching from Supabase...')

# 1. All essays
essays_raw = supabase.table('essays').select(
    'essay_id, question, structure_type, batch_id'
).execute().data
print(f'  essays          : {len(essays_raw)}')

# Report P1 batch_ids found (informational only — we use our own BATCH_ID)
p1_batch_ids = sorted({e['batch_id'] for e in essays_raw if e.get('batch_id')})
print(f'  P1 batch_ids    : {len(p1_batch_ids)} found')
for bid in p1_batch_ids:
    count = sum(1 for e in essays_raw if e.get('batch_id') == bid)
    print(f'    {bid}  ({count} essays)')

# 2. All essay-sentence links
es_raw = supabase.table('essay_sentences').select(
    'essay_id, sentence_id, paragraph_type, order'
).execute().data
print(f'  essay_sentences : {len(es_raw)}')

# 3. All sentences (with embeddings for similarity computation)
sentence_ids = list({row['sentence_id'] for row in es_raw})
sentences_raw = fetch_in_chunks(
    'sentences', 'sentence_id', sentence_ids,
    'sentence_id, canonical_text, rhetoric_tag, direction_tag, embedding'
)
print(f'  sentences       : {len(sentences_raw)}')

# 4. All lexical items (with embeddings for similarity computation)
lexical_raw = fetch_in_chunks(
    'lexical_items', 'sentence_id', sentence_ids,
    'lexical_id, sentence_id, phrase, embedding'
)
print(f'  lexical_items   : {len(lexical_raw)}')

# 5. All sentence-syntax links + syntax patterns
ss_raw = fetch_in_chunks(
    'sentence_syntax', 'sentence_id', sentence_ids,
    'sentence_id, syntax_id'
)
syntax_ids = list({row['syntax_id'] for row in ss_raw})
syntax_raw = fetch_in_chunks(
    'syntax_items', 'syntax_id', syntax_ids,
    'syntax_id, pattern'
) if syntax_ids else []
print(f'  syntax_items    : {len(syntax_raw)}')

print('\n✅ Bulk fetch complete')

Fetching from Supabase...
  essays          : 101
  P1 batch_ids    : 2 found
    88013f77-53f5-4a51-bcc7-aebe65a66229  (1 essays)
    ebc5e129-a552-426c-be13-86745cca850c  (100 essays)
  essay_sentences : 1000
  sentences       : 1000
  lexical_items   : 2000
  syntax_items    : 988

✅ Bulk fetch complete


---
## Cell 6 — Assemble essay objects

Joins the 6 raw tables into rich essay dicts. Each essay object contains:
- `essay_id`, `question`, `structure_type`
- `directions` — list of unique direction_tag strings from its sentences
- `sentences` — list of sentence dicts, each with lexical_items and syntax_items

In [6]:
from collections import defaultdict

PARA_ORDER = ['introduction', 'body_1', 'body_2', 'body_3', 'conclusion']

# Build lookup tables
sentences_by_id = {s['sentence_id']: s for s in sentences_raw}
syntax_by_id    = {s['syntax_id']: s['pattern'] for s in syntax_raw}

# lexical_items: sentence_id -> list of {lexical_id, phrase, embedding}
lexical_by_sent = defaultdict(list)
for row in lexical_raw:
    lexical_by_sent[row['sentence_id']].append({
        'lexical_id': row['lexical_id'],
        'phrase':     row['phrase'],
        'embedding':  row.get('embedding'),
    })

# syntax_items: sentence_id -> list of pattern strings
syntax_by_sent = defaultdict(list)
for row in ss_raw:
    pattern = syntax_by_id.get(row['syntax_id'])
    if pattern:
        syntax_by_sent[row['sentence_id']].append(pattern)

# Group essay_sentences by essay_id
es_by_essay = defaultdict(list)
for row in es_raw:
    es_by_essay[row['essay_id']].append(row)

# Assemble rich essay objects
essay_objects = {}

for e in essays_raw:
    eid   = e['essay_id']
    links = sorted(
        es_by_essay[eid],
        key=lambda r: (
            PARA_ORDER.index(r['paragraph_type'])
            if r['paragraph_type'] in PARA_ORDER else 99,
            r['order']
        )
    )

    sentences = []
    for link in links:
        sid = link['sentence_id']
        s   = sentences_by_id.get(sid)
        if not s:
            continue
        sentences.append({
            'sentence_id':    sid,
            'paragraph_type': link['paragraph_type'],
            'order':          link['order'],
            'canonical_text': s['canonical_text'],
            'rhetoric_tag':   s['rhetoric_tag'],
            'direction_tag':  s['direction_tag'],
            'lexical_items':  lexical_by_sent.get(sid, []),
            'syntax_items':   syntax_by_sent.get(sid, []),
            'embedding':      s.get('embedding'),   # kept for similarity; stripped at export
        })

    # Derive essay-level directions from sentence direction_tags
    directions = list({s['direction_tag'] for s in sentences if s['direction_tag']})

    essay_objects[eid] = {
        'essay_id':       eid,
        'question':       e['question'],
        'structure_type': e['structure_type'],
        'directions':     directions,
        'sentences':      sentences,
    }

total_sentences = sum(len(e['sentences']) for e in essay_objects.values())
total_lex       = sum(
    len(s['lexical_items'])
    for e in essay_objects.values()
    for s in e['sentences']
)
print(f'✅ Assembled {len(essay_objects)} essay objects')
print(f'   Total sentences    : {total_sentences}')
print(f'   Total lexical items: {total_lex}')
print(f'   Avg per essay      : {total_sentences / len(essay_objects):.1f} sentences')

✅ Assembled 101 essay objects
   Total sentences    : 1000
   Total lexical items: 2000
   Avg per essay      : 9.9 sentences


---
## Cell 7 — Inspect dataset

Quick overview of essay and direction counts before running the greedy algorithm.

In [7]:
from collections import Counter

all_directions = set()
for e in essay_objects.values():
    all_directions.update(e['directions'])

dir_counts = Counter()
for e in essay_objects.values():
    dir_counts.update(e['directions'])

total_Q = len(essay_objects)

print(f'Total essays           : {total_Q}')
print(f'Unique direction tags  : {len(all_directions)}')
print(f'Tier 50 target         : {round(total_Q * 0.50)} essays')
print(f'Tier 80 target         : {round(total_Q * 0.80)} essays')
print(f'\nTop 10 directions by essay coverage:')
for direction, count in dir_counts.most_common(10):
    print(f'  {direction:<50} {count:>4} essays')

Total essays           : 101
Unique direction tags  : 40
Tier 50 target         : 50 essays
Tier 80 target         : 81 essays

Top 10 directions by essay coverage:
  sustainable_collective_forward                       15 essays
  material_sustainable_neg_forward                     13 essays
  individual_progress_reverse                          10 essays
  progress_preservation_displacement                   10 essays
  material_progress_reverse                            10 essays
  material_individual_forward                           8 essays
  individual_collective_pos_forward                     7 essays
  collective_individual_enabling                        7 essays
  material_progress_forward                             6 essays
  progress_preservation_enabling                        6 essays


---
## Cell 8 — Greedy set-cover algorithm

The core algorithm that selects directions and model essays for each tier.

**How it works:**
1. Pick the direction that covers the most uncovered essays
2. Select the most focused model essay (fewest direction tags = purest concept)
3. Repeat until target coverage reached

**This is the same algorithm as v2 — no logic changes.**

In [ ]:
def greedy_cover_with_pairing(essay_objects_dict, target_count):
    """
    Run greedy set-cover on essays by direction_tag.
    Returns: (directions_used: set, covered_ids: set, steps: list)

    Each step selects the direction that covers the most new essays,
    then picks the model essay (richest — most direction tags, spiral tiebreak).
    """
    essays  = list(essay_objects_dict.values())
    all_dir = set()
    for e in essays:
        all_dir.update(e['directions'])

    S           = set()   # selected direction tags
    covered_ids = set()   # essay_ids covered so far
    steps       = []
    learned_set = set()   # cumulative directions from all model essays picked so far

    while len(covered_ids) < target_count:

        # Pick the direction that covers the most new essays
        best_dir     = None
        best_new_ids = set()

        for d in all_dir:
            if d in S:
                continue
            new = {
                e['essay_id'] for e in essays
                if e['essay_id'] not in covered_ids and d in e['directions']
            }
            if len(new) > len(best_new_ids):
                best_new_ids = new
                best_dir     = d

        if best_dir is None or not best_new_ids:
            print('  ⚠️  Cannot reach target — stopping early.')
            break

        S.add(best_dir)
        covered_ids.update(best_new_ids)

        # Model = richest essay (most direction tags), spiral tiebreak (most overlap with learned)
        new_essays = [essay_objects_dict[eid] for eid in best_new_ids]
        model      = max(
            new_essays,
            key=lambda e: (
                len(e['directions']),                      # primary: richest
                len(set(e['directions']) & learned_set),   # secondary: most overlap with learned
            )
        )

        # Update learned_set AFTER picking model (so prior steps inform the tiebreak)
        learned_set.update(model['directions'])

        steps.append({
            'step':           len(steps) + 1,
            'new_directions': [best_dir],
            'model':          model,
            'newly_covered':  len(best_new_ids),
            'total_covered':  len(covered_ids),
            'pct':            round(len(covered_ids) / len(essays) * 100, 1),
        })

    return S, covered_ids, steps


print('✅ Greedy function ready')

---
## Cell 9 — Run greedy for tier_50 and tier_80

In [9]:
results = {}

for pct, label in [(0.50, 'tier_50'), (0.80, 'tier_80')]:
    target = round(total_Q * pct)
    S, covered_ids, steps = greedy_cover_with_pairing(essay_objects, target)
    results[label] = {
        'directions_used': S,
        'covered_ids':     covered_ids,
        'steps':           steps,
    }

    print('=' * 72)
    print(f'{label.upper()} — target {target}/{total_Q} ({pct*100:.0f}%)')
    print('=' * 72)
    print(f"{'#':<4} {'Direction':<42} {'New':>4} {'Tot':>4} {'%':>5}  {'Model (truncated)'}")
    print('-' * 80)
    for s in steps:
        model_q = (s['model']['question'][:40] + '…') if s['model'] else '—'
        print(
            f"{s['step']:<4} {s['new_directions'][0]:<42} "
            f"{s['newly_covered']:>4} {s['total_covered']:>4} {s['pct']:>4}%  "
            f"{model_q}"
        )
    print(f"\nDirections selected : {len(S)}")
    print(f"Essays covered      : {len(covered_ids)}/{total_Q}\n")

TIER_50 — target 50/101 (50%)
#    Direction                                   New  Tot     %  Model (truncated)
--------------------------------------------------------------------------------
1    sustainable_collective_forward               15   15 14.9%  Access to clean water is a basic human r…
2    progress_preservation_displacement           10   25 24.8%  Some people today argue that media has b…
3    material_sustainable_neg_forward             10   35 34.7%  In recent years, there has been a rise i…
4    individual_progress_reverse                   7   42 41.6%  Some employers are giving more value to …
5    material_progress_reverse                     6   48 47.5%  Developing countries should be encourage…
6    material_individual_forward                   5   53 52.5%  Some people think that government should…

Directions selected : 6
Essays covered      : 53/101

  ⚠️  Cannot reach target — stopping early.
TIER_80 — target 81/101 (80%)
#    Direction                     

---
## Cell 10 — Compute Q Bank unlocks (enhanced)

**New in v3:** Each Q Bank unlock now includes `shared_directions` — the full list
of direction_ids that overlap between the Q Bank essay and the unlocking model essay.
This lets the School UI show *why* a question relates to the unlocking unit.

In [ ]:
def compute_qbank_unlocks(essay_objects_dict, steps, covered_ids):
    """
    For each Q Bank essay (covered but not a model), find the earliest
    sequence step at which the cumulative set of learned directions
    overlaps with at least 2 of the essay's directions.

    'Learned directions' = union of all model essay directions up to that step.

    Returns: {
        essay_id: {
            'unlock_position':   int (1-indexed),
            'shared_directions': [direction_id, ...]  # the ≥2 tags that triggered unlock
        }
    }
    If a Q Bank essay never reaches ≥2 overlap, it is omitted (permanently locked).
    """
    model_ids = {s['model']['essay_id'] for s in steps}
    qbank_ids = covered_ids - model_ids
    unlocks   = {}

    for qbank_id in qbank_ids:
        qbank_dirs         = set(essay_objects_dict[qbank_id]['directions'])
        cumulative_learned = set()
        unlock_pos         = None
        shared             = []

        for step in steps:
            cumulative_learned.update(step['model']['directions'])
            overlap = qbank_dirs & cumulative_learned
            if len(overlap) >= 2:
                unlock_pos = step['step']
                shared     = sorted(list(overlap))  # sorted for determinism
                break

        if unlock_pos is None:
            # Essay never reaches ≥2 direction overlap — trust data contract, leave locked
            print(f'  ⚠️  {qbank_id[:8]}… never reaches ≥2 direction overlap — omitted from Q Bank')
            continue

        unlocks[qbank_id] = {
            'unlock_position':   unlock_pos,
            'shared_directions': shared,
        }

    return unlocks


# Run for each tier
print('Computing Q Bank unlock positions...')
for label in ['tier_50', 'tier_80']:
    unlocks = compute_qbank_unlocks(
        essay_objects,
        results[label]['steps'],
        results[label]['covered_ids'],
    )
    results[label]['qbank_unlocks'] = unlocks
    pool_size  = len(results[label]['covered_ids'])
    seq_size   = len(results[label]['steps'])
    qbank_size = len(unlocks)
    print(f'  {label}: pool={pool_size}  sequence={seq_size}  Q Bank={qbank_size}')

    # Show shared_directions stats
    with_shared = sum(1 for v in unlocks.values() if v['shared_directions'])
    avg_shared  = sum(len(v['shared_directions']) for v in unlocks.values()) / max(len(unlocks), 1)
    print(f'    {with_shared}/{qbank_size} have shared_directions (avg {avg_shared:.1f} per essay)')

print('\n✅ Q Bank unlock positions ready')

---
## Cell 11 — Build similarity indices

Three pre-computed similarity indices per tier:
1. **Sentence similarity** — top-20 most similar sentences (used by P3 for MCQ distractor ranking)
2. **Lexical similarity** (cosine 0.5–0.7) — similar phrases for L1F fill practice
3. **Hint sentences** (cosine 0.7–0.9) — recall cues for L2F sentence fill practice

**Same algorithms as v2 — no logic changes.**

In [11]:
import json
import numpy as np


def build_similarity_index(essay_objects_dict, covered_ids, top_n=20):
    """
    Per-sentence similarity index. Three sub-lists per sentence:
      overall       → top_n most similar (no tag constraint)
      same_rhetoric → top_n sharing the same rhetoric_tag
      diff_rhetoric → top_n with a different rhetoric_tag
    """
    all_sentences = []
    for eid in covered_ids:
        all_sentences.extend(essay_objects_dict[eid]['sentences'])

    if not all_sentences or all_sentences[0].get('embedding') is None:
        print('  ⚠️  No sentence embeddings — sentence similarity index skipped')
        return {}

    ids    = [s['sentence_id']  for s in all_sentences]
    r_tags = [s['rhetoric_tag'] for s in all_sentences]

    embeddings = np.array([
        json.loads(s['embedding']) if isinstance(s['embedding'], str) else s['embedding']
        for s in all_sentences
    ], dtype=float)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1
    normalised = embeddings / norms
    sim_matrix = normalised @ normalised.T

    index = {}
    for i, sid in enumerate(ids):
        row    = sim_matrix[i].copy()
        row[i] = -1
        ranked = np.argsort(row)[::-1]

        this_r = r_tags[i]
        index[sid] = {
            'overall':       [ids[j] for j in ranked[:top_n]],
            'same_rhetoric': [ids[j] for j in ranked if r_tags[j] == this_r][:top_n],
            'diff_rhetoric': [ids[j] for j in ranked if r_tags[j] != this_r][:top_n],
        }
    return index


def build_lexical_similarity_index(essay_objects_dict, covered_ids,
                                   sim_min=0.5, sim_max=0.7):
    """
    Per-lexical-item distractor candidates for L1F (phrase fill).
    Keeps only candidates with cosine in [0.5, 0.7] range.
    Returns: { lexical_id: [lexical_id, ...] }
    """
    all_lex = []
    for eid in covered_ids:
        for s in essay_objects_dict[eid]['sentences']:
            for li in s.get('lexical_items', []):
                if li.get('embedding') is not None:
                    all_lex.append(li)

    if not all_lex:
        print('  ⚠️  No lexical embeddings — lexical similarity index skipped')
        return {}

    ids = [li['lexical_id'] for li in all_lex]
    embeddings = np.array([
        json.loads(li['embedding']) if isinstance(li['embedding'], str) else li['embedding']
        for li in all_lex
    ], dtype=float)

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1
    normalised = embeddings / norms
    sim_matrix = normalised @ normalised.T

    index = {}
    N = len(ids)
    for i, lid in enumerate(ids):
        row    = sim_matrix[i].copy()
        row[i] = -1
        candidates = [
            (ids[j], float(row[j]))
            for j in range(N)
            if j != i and sim_min <= row[j] <= sim_max
        ]
        candidates.sort(key=lambda x: x[1], reverse=True)
        index[lid] = [lid for lid, _ in candidates]
    return index


def build_hint_sentences_index(essay_objects_dict, covered_ids,
                                sim_min=0.7, sim_max=0.9):
    """
    Per-sentence hint sentences for L2F (sentence fill).
    Finds semantically similar sentences from OTHER essays (same-essay excluded)
    within cosine [0.7, 0.9]. No top-N cap. Returns canonical_text strings.
    """
    all_sentences = []
    for eid in covered_ids:
        for s in essay_objects_dict[eid]['sentences']:
            all_sentences.append({**s, '_essay_id': eid})

    if not all_sentences or all_sentences[0].get('embedding') is None:
        print('  ⚠️  No sentence embeddings — hint_sentences index skipped')
        return {}

    ids       = [s['sentence_id']   for s in all_sentences]
    essay_ids = [s['_essay_id']     for s in all_sentences]
    texts     = [s['canonical_text'] for s in all_sentences]

    embeddings = np.array([
        json.loads(s['embedding']) if isinstance(s['embedding'], str) else s['embedding']
        for s in all_sentences
    ], dtype=float)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1
    normalised = embeddings / norms
    sim_matrix = normalised @ normalised.T

    index = {}
    N = len(ids)
    for i, sid in enumerate(ids):
        row    = sim_matrix[i].copy()
        row[i] = -1
        candidates = [
            (texts[j], float(row[j]))
            for j in range(N)
            if j != i
            and essay_ids[j] != essay_ids[i]
            and sim_min <= row[j] <= sim_max
        ]
        candidates.sort(key=lambda x: x[1], reverse=True)
        index[sid] = [text for text, _ in candidates]
    return index


# Run all three indices for each tier
print('Building similarity indices (this may take a minute)...')
for label in ['tier_50', 'tier_80']:
    covered = results[label]['covered_ids']

    sent_idx = build_similarity_index(essay_objects, covered, top_n=20)
    results[label]['similarity_index'] = sent_idx
    print(f'  {label} sentence index     : {len(sent_idx)} sentences')

    lex_idx = build_lexical_similarity_index(essay_objects, covered)
    results[label]['lexical_similarity_index'] = lex_idx
    n_with = sum(1 for v in lex_idx.values() if v)
    print(f'  {label} lexical index      : {len(lex_idx)} items, {n_with} with candidates')

    hint_idx = build_hint_sentences_index(essay_objects, covered)
    results[label]['hint_sentences_index'] = hint_idx
    n_hints = sum(1 for v in hint_idx.values() if v)
    avg_h   = sum(len(v) for v in hint_idx.values()) / max(len(hint_idx), 1)
    print(f'  {label} hint_sentences     : {len(hint_idx)} sentences, {n_hints} with hints (avg {avg_h:.1f})')

print('\n✅ Similarity indices ready')

Building similarity indices (this may take a minute)...
  tier_50 sentence index     : 680 sentences
  tier_50 lexical index      : 1360 items, 1279 with candidates
  tier_50 hint_sentences     : 680 sentences, 109 with hints (avg 0.3)
  tier_80 sentence index     : 978 sentences
  tier_80 lexical index      : 1956 items, 1854 with candidates
  tier_80 hint_sentences     : 978 sentences, 157 with hints (avg 0.4)

✅ Similarity indices ready


---
## Cell 12 — DIRECTION_LOOKUP (126 entries)

Maps direction IDs → `{argument, logic, blog_url}`. Source: Reference-v2.md Section 1. All 126 entries across 21 pairs of 7 dimensions.

**v5 change from v3:** Replaces 39 old-name entries with 126 new-name entries matching P1 v5 `VALID_DIRECTION_TAGS`. Values expanded from flat string to `{argument, logic, blog_url}` per LAB-SCHOOL CONTRACT v3.

In [12]:
# All 126 direction entries from Reference-v2.md Section 1
# Each entry: direction_tag → {"argument": ..., "logic": ..., "blog_url": None}
# blog_url is null until blog post is published
DIRECTION_LOOKUP = {
    # ── Pair 1: MATERIAL ↔ SUSTAINABLE (8) ──
    "material_sustainable_causal_neg": {
        "argument": "Economic growth damages the environment",
        "logic": "When businesses and countries prioritize profit and output, they consume resources and produce pollution faster than the planet can recover.",
        "blog_url": None,
    },
    "sustainable_material_causal_neg": {
        "argument": "Environmental protection limits economic growth",
        "logic": "Rules designed to protect nature — emissions limits, resource caps, clean energy requirements — raise costs and slow down industries.",
        "blog_url": None,
    },
    "material_sustainable_tradeoff": {
        "argument": "Countries have to choose between growing their economy and protecting the environment",
        "logic": "Developing nations cannot always afford clean alternatives. The choice between a coal plant and no electricity is real for millions of people.",
        "blog_url": None,
    },
    "material_sustainable_synergy": {
        "argument": "A green economy proves growth and sustainability work together",
        "logic": "Solar and wind energy now cost less than fossil fuels. Clean technology creates jobs. Protecting nature preserves the resources economies depend on.",
        "blog_url": None,
    },
    "material_sustainable_instrument": {
        "argument": "Economic growth funds environmental protection",
        "logic": "Richer countries can afford cleaner technology, stronger regulations, and environmental restoration. Poverty makes sustainability harder, not easier.",
        "blog_url": None,
    },
    "material_sustainable_blocking": {
        "argument": "Short-term profit logic makes sustainability impossible",
        "logic": "Businesses that must show quarterly profits cannot plan for 50-year environmental timelines. The incentive structure works against long-term thinking by design.",
        "blog_url": None,
    },
    "material_sustainable_transformation": {
        "argument": "What looks like economic growth is actually long-term loss",
        "logic": "Cutting down a forest counts as GDP growth. The cost of losing that forest — flood protection, air quality, biodiversity — never appears on the balance sheet.",
        "blog_url": None,
    },
    "material_sustainable_feedback_neg": {
        "argument": "Economic growth and environmental damage feed each other",
        "logic": "More production requires more resources, which causes more damage, which requires more production to pay for the consequences. The cycle compounds over time.",
        "blog_url": None,
    },
    # ── Pair 2: MATERIAL ↔ INDIVIDUAL (6) ──
    "individual_material_causal_pos": {
        "argument": "Personal freedom drives economic growth",
        "logic": "When people can start businesses, keep their earnings, and make their own economic decisions, economies grow faster and create more wealth.",
        "blog_url": None,
    },
    "material_individual_causal_neg": {
        "argument": "Wealth concentration reduces real freedom for most people",
        "logic": "When economic success builds on itself, a small group ends up controlling most resources. Freedom without money is limited in practice.",
        "blog_url": None,
    },
    "material_individual_synergy": {
        "argument": "Free markets and personal freedom strengthen each other",
        "logic": "A person who can open a shop without government permission earns money, hires neighbours, and creates competition that lowers prices for everyone. The shop owner's freedom becomes the community's prosperity.",
        "blog_url": None,
    },
    "material_individual_instrument": {
        "argument": "Economic security is what makes personal freedom real",
        "logic": "Legal rights mean little when people are struggling to survive. Financial stability is what turns freedom on paper into freedom in practice.",
        "blog_url": None,
    },
    "material_individual_tradeoff": {
        "argument": "Protecting individual economic freedom creates inequality",
        "logic": "Letting people keep what they earn and build what they want produces vastly different outcomes. The freedom that benefits winners disadvantages those who start with less.",
        "blog_url": None,
    },
    "material_individual_feedback_pos": {
        "argument": "Wealth and freedom reinforce each other",
        "logic": "A family with savings can choose better schools, which leads to better jobs, which builds more savings. A family without savings has no choice at any step. The same system accelerates the rich and traps the poor.",
        "blog_url": None,
    },
    # ── Pair 3: MATERIAL ↔ COLLECTIVE (6) ──
    "material_collective_instrument": {
        "argument": "Economic growth funds public services",
        "logic": "Schools, hospitals, and infrastructure all require tax revenue. A productive economy is what makes collective investment possible in the first place.",
        "blog_url": None,
    },
    "collective_material_instrument": {
        "argument": "Public investment drives economic growth",
        "logic": "Roads, education, legal systems, and stable governance are what businesses depend on. The private sector grows on a foundation the government builds.",
        "blog_url": None,
    },
    "material_collective_tradeoff": {
        "argument": "Taxation and regulation reduce economic productivity",
        "logic": "Every rule businesses must follow and every tax they pay is a cost. Too much collective intervention slows down the engine that generates wealth.",
        "blog_url": None,
    },
    "collective_material_blocking": {
        "argument": "Without redistribution, economic growth benefits only the few",
        "logic": "Markets left alone concentrate wealth. Without collective mechanisms — taxes, welfare, labor laws — growth never reaches the people who need it most.",
        "blog_url": None,
    },
    "material_collective_synergy": {
        "argument": "Prosperous economies and strong societies build each other",
        "logic": "A country that taxes profits to fund schools produces educated workers who build better companies that generate more tax revenue. Scandinavia runs on this cycle — high taxes, high skills, high GDP per capita.",
        "blog_url": None,
    },
    "material_collective_feedback_neg": {
        "argument": "Wealth and political power reinforce each other",
        "logic": "Economic winners use their resources to shape policy in their favor. Favorable policy helps them win more. The gap between rich and poor widens over time.",
        "blog_url": None,
    },
    # ── Pair 4: MATERIAL ↔ PROGRESS (6) ──
    "material_progress_instrument": {
        "argument": "Economic wealth funds innovation",
        "logic": "Research, development, and experimentation cost money. Rich societies and profitable companies are the ones that can afford to invest in new technology.",
        "blog_url": None,
    },
    "progress_material_causal_pos": {
        "argument": "Innovation drives economic growth",
        "logic": "Every major economic boom in history was built on new technology. The internet, electricity, and steam power all created enormous new wealth.",
        "blog_url": None,
    },
    "material_progress_feedback_pos": {
        "argument": "Wealth and innovation fuel each other",
        "logic": "Profits fund research. Research creates products. Products generate profits. The cycle explains why technological and economic advancement happen together.",
        "blog_url": None,
    },
    "material_progress_tradeoff": {
        "argument": "Short-term profit pressure kills long-term innovation",
        "logic": "Genuine breakthroughs take decades and don't guarantee returns. Businesses focused on quarterly results consistently underinvest in the research that matters most.",
        "blog_url": None,
    },
    "progress_material_spillover_neg": {
        "argument": "Innovation destroys jobs faster than it creates them",
        "logic": "New technology replaces workers immediately. The new jobs it eventually creates are fewer, different, and not available to the people displaced. The gap is real and painful.",
        "blog_url": None,
    },
    "material_progress_transformation": {
        "argument": "What investors call disruption, workers call destruction",
        "logic": "An automated factory is a productivity gain on a spreadsheet and a community collapse for the town that depended on it. Both descriptions are accurate at the same time.",
        "blog_url": None,
    },
    # ── Pair 5: MATERIAL ↔ PRESERVATION (6) ──
    "material_preservation_causal_neg": {
        "argument": "Economic development destroys traditional culture",
        "logic": "When money flows into a community, it changes what people do, value, and practice. Traditional ways of life rarely survive full commercialization intact.",
        "blog_url": None,
    },
    "material_preservation_instrument_pos": {
        "argument": "Economic value gives traditions a reason to survive",
        "logic": "Crafts, foods, and cultural practices that generate income get maintained. Those that don't generate income disappear. Money is a crude but effective preservation tool.",
        "blog_url": None,
    },
    "material_preservation_tradeoff": {
        "argument": "Communities have to choose between economic growth and cultural identity",
        "logic": "Development brings jobs and income but also replaces local practices with global ones. The gain and the loss are real and often happen at the same time.",
        "blog_url": None,
    },
    "material_preservation_paradox": {
        "argument": "Selling culture saves it and destroys it at the same time",
        "logic": "Tourism and commercial interest fund the maintenance of traditions. But traditions performed for money and tourists gradually lose the meaning that made them worth preserving.",
        "blog_url": None,
    },
    "preservation_material_instrument": {
        "argument": "Cultural heritage drives economic activity",
        "logic": "Tourism, local food, traditional crafts, and cultural festivals generate significant income for communities. Preserving culture is also preserving an economic asset.",
        "blog_url": None,
    },
    "material_preservation_spillover_neg": {
        "argument": "Economic pressure accidentally erases cultural practices",
        "logic": "No one decides to destroy a tradition. But when the economy rewards different skills and schedules, the conditions that sustained traditions quietly disappear.",
        "blog_url": None,
    },
    # ── Pair 6: MATERIAL ↔ FLOURISHING (7) ──
    "material_flourishing_causal_neg": {
        "argument": "Economic pressure harms human wellbeing",
        "logic": "When work is organized around maximum output, people experience stress, loss of meaning, and exhaustion as normal features of daily life.",
        "blog_url": None,
    },
    "material_flourishing_causal_pos": {
        "argument": "Financial security improves human wellbeing",
        "logic": "Poverty, debt, and economic insecurity cause enormous psychological suffering. Having enough money removes a major source of anxiety and opens up real choices.",
        "blog_url": None,
    },
    "material_flourishing_tradeoff": {
        "argument": "People have to choose between higher income and more meaningful lives",
        "logic": "Better-paying jobs often demand more time, more stress, and less autonomy. The salary and the life people actually want frequently pull in opposite directions.",
        "blog_url": None,
    },
    "material_flourishing_instrument": {
        "argument": "Money is only worth having if it improves your life",
        "logic": "Income is not the goal. A better quality of life is the goal. When higher earnings come at the cost of health, relationships, and meaning, the trade is a bad one.",
        "blog_url": None,
    },
    "flourishing_material_instrument": {
        "argument": "Happy, fulfilled workers are more economically productive",
        "logic": "A nurse who feels valued stays 10 years and mentors others. A nurse treated as a cost centre burns out in 18 months. The hospital spends more replacing burned-out staff than it would investing in wellbeing.",
        "blog_url": None,
    },
    "material_flourishing_blocking": {
        "argument": "Systems built around profit leave no room for human dignity",
        "logic": "When every decision is made on economic grounds, things that matter intrinsically — care, rest, creativity, conscience — get treated as costs to be minimized.",
        "blog_url": None,
    },
    "material_flourishing_feedback_neg": {
        "argument": "Economic insecurity and poor wellbeing trap each other",
        "logic": "Financial stress damages mental health. Poor mental health reduces earning ability. Lower earnings increase financial stress. The cycle is hard to break from inside.",
        "blog_url": None,
    },
    # ── Pair 7: SUSTAINABLE ↔ INDIVIDUAL (6) ──
    "sustainable_individual_causal_pos": {
        "argument": "A healthy environment directly improves individual lives",
        "logic": "Clean air, safe water, and stable climate affect people's physical health, where they can live, and what choices are available to them every day.",
        "blog_url": None,
    },
    "individual_sustainable_causal_neg": {
        "argument": "Individual consumption choices damage the environment",
        "logic": "Billions of small decisions — what to eat, how to travel, what to buy — add up to enormous environmental impact. Individual behavior is not trivial at scale.",
        "blog_url": None,
    },
    "individual_sustainable_insufficient": {
        "argument": "Individual action alone cannot solve environmental problems",
        "logic": "The math doesn't work. Personal choices operate at a scale too small to address industrial-level damage. Systemic problems need systemic solutions, not just willing individuals.",
        "blog_url": None,
    },
    "sustainable_individual_instrument": {
        "argument": "Environmental protection is ultimately about protecting people",
        "logic": "Saving ecosystems is not about nature for its own sake. It is about preserving the conditions humans need to live, eat, and survive across generations.",
        "blog_url": None,
    },
    "individual_sustainable_tradeoff": {
        "argument": "Sustainable choices cost individuals more",
        "logic": "Ethical products, cleaner transport, and lower-consumption lifestyles are consistently more expensive and less convenient than the alternatives. Virtue has a real price.",
        "blog_url": None,
    },
    "sustainable_individual_spillover": {
        "argument": "Environmental damage harms individuals who had no part in causing it",
        "logic": "Pollution from industry affects nearby communities. Climate change affects farmers, fishermen, and the poor first. Individual harm flows from decisions others made.",
        "blog_url": None,
    },
    # ── Pair 8: SUSTAINABLE ↔ COLLECTIVE (6) ──
    "sustainable_collective_instrument": {
        "argument": "Government action is the only reliable way to protect the environment",
        "logic": "Voluntary behavior and market forces consistently underdeliver on environmental protection. Only laws, taxes, and enforcement can guarantee change at the required scale.",
        "blog_url": None,
    },
    "collective_sustainable_instrument": {
        "argument": "Environmental health is a public good governments must provide",
        "logic": "Clean air and a stable climate benefit everyone and cannot be privately owned. These are exactly the goods that collective action exists to provide.",
        "blog_url": None,
    },
    "sustainable_collective_blocking": {
        "argument": "Without collective rules, individual and market incentives destroy the environment",
        "logic": "Polluters profit while others pay the costs. Without regulation, there is no financial reason to stop. The incentive to free-ride is structurally built into the system.",
        "blog_url": None,
    },
    "collective_sustainable_causal_pos": {
        "argument": "Strong government policy produces measurable environmental improvement",
        "logic": "Countries with strict emissions laws, protected areas, and carbon pricing consistently show better environmental outcomes than those relying on voluntary action.",
        "blog_url": None,
    },
    "sustainable_collective_tradeoff": {
        "argument": "Environmental regulation reduces economic and personal freedom",
        "logic": "Every environmental rule restricts something — what industries can do, what products can be sold, how land can be used. Protection always has a cost to someone.",
        "blog_url": None,
    },
    "collective_sustainable_feedback_pos": {
        "argument": "Environmental governance and public health reinforce each other",
        "logic": "Cleaner environments reduce healthcare costs, increase productivity, and improve quality of life — giving governments more resources to invest in further environmental protection.",
        "blog_url": None,
    },
    # ── Pair 9: SUSTAINABLE ↔ PROGRESS (6) ──
    "progress_sustainable_causal_pos": {
        "argument": "Technology solves environmental problems",
        "logic": "Solar panels, electric vehicles, and precision agriculture show that innovation can deliver the same or better results with much less environmental cost.",
        "blog_url": None,
    },
    "progress_sustainable_causal_neg": {
        "argument": "Technology creates new environmental problems while solving old ones",
        "logic": "Electric cars reduce emissions but require mining. AI reduces some waste but consumes enormous energy. Every solution introduces new environmental costs alongside the benefits.",
        "blog_url": None,
    },
    "sustainable_progress_instrument": {
        "argument": "Environmental limits force better innovation",
        "logic": "Scarcity and regulation push engineers and businesses to find solutions they would never have looked for otherwise. Constraint is one of the most reliable drivers of invention.",
        "blog_url": None,
    },
    "progress_sustainable_spillover_neg": {
        "argument": "Efficiency gains lead to more consumption, not less",
        "logic": "Cheaper, cleaner technology makes people use more of it. More efficient cars led to more driving. Better engines burned more total fuel. Improvement doesn't automatically mean reduction.",
        "blog_url": None,
    },
    "sustainable_progress_tradeoff": {
        "argument": "Protecting the environment sometimes means accepting less technology",
        "logic": "Some innovations are too damaging to allow regardless of their benefits. The question of what to build is not only technical — it is also about what we can afford to lose.",
        "blog_url": None,
    },
    "progress_sustainable_transformation": {
        "argument": "What technologists call a solution, ecologists call a new problem",
        "logic": "A nuclear plant solving emissions is simultaneously an unsolved waste disposal problem. The same technology looks like progress or risk depending entirely on what you measure.",
        "blog_url": None,
    },
    # ── Pair 10: SUSTAINABLE ↔ PRESERVATION (5) ──
    "sustainable_preservation_synergy": {
        "argument": "Traditional practices and ecological survival support each other",
        "logic": "Aboriginal fire management prevented catastrophic bushfires for 60,000 years. When colonisers stopped it, forests burned uncontrollably. Protecting the cultural practice was protecting the ecosystem — same action, both outcomes.",
        "blog_url": None,
    },
    "preservation_sustainable_instrument": {
        "argument": "Traditional knowledge is a practical guide to sustainability",
        "logic": "Farming methods, water management, and land use refined over centuries frequently outperform modern industrial approaches on long-term ecological outcomes.",
        "blog_url": None,
    },
    "sustainable_preservation_causal_pos": {
        "argument": "Protecting ecosystems preserves the cultures that depend on them",
        "logic": "When rivers are polluted, the fishing village downstream loses not just food but its identity — the festivals, the knowledge, the way children learn from parents. Destroy the river and the culture dies with it.",
        "blog_url": None,
    },
    "preservation_sustainable_causal_neg": {
        "argument": "Some traditional practices damage the environment",
        "logic": "Not all inherited methods are ecologically sound. Some traditional farming, hunting, and land use practices deplete resources just as modern industrial ones do.",
        "blog_url": None,
    },
    "sustainable_preservation_tradeoff": {
        "argument": "Environmental protection sometimes conflicts with traditional land use",
        "logic": "Conservation areas restrict hunting, grazing, and farming that communities have practiced for generations. Protecting nature can directly harm the people who live in it.",
        "blog_url": None,
    },
    # ── Pair 11: SUSTAINABLE ↔ FLOURISHING (5) ──
    "sustainable_flourishing_causal_pos": {
        "argument": "A healthy environment supports human wellbeing",
        "logic": "Children who grow up near green spaces have measurably lower anxiety and better concentration. Adults who breathe polluted air have higher rates of depression. The environment is not background — it is a direct input into mental health.",
        "blog_url": None,
    },
    "flourishing_sustainable_instrument": {
        "argument": "Human wellbeing depends on ecological health",
        "logic": "You cannot have thriving people on a dying planet. Long-term human flourishing and environmental sustainability are not separate goals — one is a condition for the other.",
        "blog_url": None,
    },
    "sustainable_flourishing_synergy": {
        "argument": "Environmental health and human wellbeing reinforce each other",
        "logic": "A community that plants urban forests sees lower heat stress, which improves residents' sleep and mood, which increases civic participation, which leads to more environmental projects. The cycle works because healthy places make healthy people who protect their places.",
        "blog_url": None,
    },
    "sustainable_flourishing_tradeoff": {
        "argument": "Environmental responsibility asks people to sacrifice present comfort",
        "logic": "Eating less meat, flying less, and consuming less require real changes to how people live. The wellbeing cost of those sacrifices is real, not just theoretical.",
        "blog_url": None,
    },
    "flourishing_sustainable_causal_neg": {
        "argument": "Prioritizing human comfort damages the environment",
        "logic": "The lifestyle choices that make people feel good — warm homes, frequent travel, abundant food — are often the ones with the highest environmental cost.",
        "blog_url": None,
    },
    # ── Pair 12: INDIVIDUAL ↔ COLLECTIVE (7) ──
    "individual_collective_tradeoff": {
        "argument": "Individual freedom and collective welfare pull against each other",
        "logic": "Rules that protect everyone restrict someone. Freedoms that benefit individuals often impose costs on others. This tension never fully resolves — it only gets managed.",
        "blog_url": None,
    },
    "collective_individual_instrument": {
        "argument": "Collective systems exist to protect individual freedom",
        "logic": "Laws, public services, and social safety nets are not the opposite of individual freedom. They are what makes freedom real for people who would otherwise have none.",
        "blog_url": None,
    },
    "individual_collective_blocking": {
        "argument": "Individual action alone cannot solve shared problems",
        "logic": "When a problem affects everyone and requires everyone's participation, voluntary individual action consistently underdelivers. Some solutions only work if everyone is in.",
        "blog_url": None,
    },
    "collective_individual_blocking": {
        "argument": "Too much collective control destroys individual autonomy",
        "logic": "When governments or communities dictate too many choices, individuals lose the ability to direct their own lives. Control justified by the common good can become oppression.",
        "blog_url": None,
    },
    "individual_collective_synergy": {
        "argument": "Strong individuals build strong communities",
        "logic": "A doctor who trained through personal ambition serves hundreds of patients. A volunteer who developed leadership skills at work organises the neighbourhood response during a flood. Individual capability becomes collective strength when it is needed most.",
        "blog_url": None,
    },
    "collective_individual_causal_pos": {
        "argument": "Social support enables individual success",
        "logic": "The entrepreneur who \"built it alone\" drove on public roads, hired publicly educated workers, and relied on courts to enforce contracts. Remove any one of those and the business never exists. Individual success stands on a collective foundation.",
        "blog_url": None,
    },
    "individual_collective_feedback_neg": {
        "argument": "Inequality and weak public systems feed each other",
        "logic": "When public systems are weak, people who can afford private alternatives opt out. When they opt out, public systems weaken further. The gap between those inside and outside widens over time.",
        "blog_url": None,
    },
    # ── Pair 13: INDIVIDUAL ↔ PROGRESS (6) ──
    "progress_individual_causal_pos": {
        "argument": "Technology expands individual freedom",
        "logic": "A farmer in rural Kenya with a smartphone can check crop prices, access weather data, and sell directly to buyers — choices that required a middleman, a library, and a radio station a generation ago. One device replaced three gatekeepers.",
        "blog_url": None,
    },
    "individual_progress_causal_pos": {
        "argument": "Personal freedom drives innovation",
        "logic": "People experimenting freely, pursuing unconventional ideas, and taking personal risks are the actual source of most breakthroughs. Control and conformity produce maintenance, not invention.",
        "blog_url": None,
    },
    "progress_individual_paradox": {
        "argument": "Technology promises freedom but creates new forms of control",
        "logic": "Platforms designed for connection harvest data, shape attention, and determine what people see. The more people rely on these tools, the less free they are from those who own them.",
        "blog_url": None,
    },
    "progress_individual_spillover_neg": {
        "argument": "Technology designed for productivity harms individual autonomy",
        "logic": "Tools built to make work more efficient end up monitoring workers, measuring output, and removing the judgment calls that gave work its human character.",
        "blog_url": None,
    },
    "progress_individual_instrument": {
        "argument": "Technology is only worth developing if it serves individual needs",
        "logic": "The point of innovation is not innovation itself. The question is whether it gives real people more control over their own lives or less.",
        "blog_url": None,
    },
    "individual_progress_instrument": {
        "argument": "Individual freedom is the condition that makes innovation possible",
        "logic": "Societies that allow people to question, experiment, and fail produce far more innovation than societies that enforce conformity, regardless of how much they invest in technology.",
        "blog_url": None,
    },
    # ── Pair 14: INDIVIDUAL ↔ PRESERVATION (5) ──
    "individual_preservation_causal_pos": {
        "argument": "Personal choices keep traditions alive",
        "logic": "Traditions survive when individuals choose to practice them, teach them, and pass them on. Without that personal commitment, no institutional support can substitute for it.",
        "blog_url": None,
    },
    "preservation_individual_causal_pos": {
        "argument": "Cultural roots give individuals identity and stability",
        "logic": "People who have a strong sense of where they come from and what they belong to navigate change, loss, and uncertainty more successfully than those without that grounding.",
        "blog_url": None,
    },
    "individual_preservation_insufficient": {
        "argument": "Individual commitment alone cannot save a dying tradition",
        "logic": "When economic pressure, social change, and generational shift all move against a practice, individual devotion slows the decline but rarely reverses it. Structure is needed.",
        "blog_url": None,
    },
    "individual_preservation_tradeoff": {
        "argument": "Personal freedom and cultural belonging pull against each other",
        "logic": "Individuals who move away from their community, adopt different values, or reject inherited practices are exercising freedom while weakening the tradition they leave behind.",
        "blog_url": None,
    },
    "preservation_individual_instrument": {
        "argument": "Cultural identity is what makes individual freedom meaningful",
        "logic": "Freedom without any roots or belonging is disorienting. People need a sense of identity to make choices that feel like their own rather than arbitrary.",
        "blog_url": None,
    },
    # ── Pair 15: INDIVIDUAL ↔ FLOURISHING (5) ──
    "individual_flourishing_synergy": {
        "argument": "Personal freedom and human wellbeing support each other",
        "logic": "A person who chooses their own career path — even a riskier one — reports more life satisfaction than someone placed in a stable job they never wanted. The act of choosing, not just the outcome, is what produces meaning.",
        "blog_url": None,
    },
    "individual_flourishing_tradeoff": {
        "argument": "Unlimited personal freedom can damage individual wellbeing",
        "logic": "The freedom to make any choice includes the freedom to make harmful ones. Without any external structure, some people consistently choose against their own long-term wellbeing.",
        "blog_url": None,
    },
    "flourishing_individual_instrument": {
        "argument": "Wellbeing is what personal freedom is ultimately for",
        "logic": "Freedom matters because it allows people to build lives that actually satisfy them. Freedom that produces anxiety, emptiness, or harm has failed at its own purpose.",
        "blog_url": None,
    },
    "individual_flourishing_causal_neg": {
        "argument": "Excessive self-focus damages wellbeing",
        "logic": "Cultures that treat individual achievement and self-optimization as the highest values consistently produce high rates of anxiety, loneliness, and dissatisfaction.",
        "blog_url": None,
    },
    "flourishing_individual_blocking": {
        "argument": "Poor mental health removes real freedom even when legal freedom exists",
        "logic": "A person trapped by addiction, severe depression, or chronic anxiety is not free in any meaningful sense, regardless of what rights they technically possess.",
        "blog_url": None,
    },
    # ── Pair 16: COLLECTIVE ↔ PROGRESS (6) ──
    "collective_progress_instrument": {
        "argument": "Government investment drives breakthroughs private markets won't fund",
        "logic": "The internet, GPS, and mRNA vaccines were all publicly funded. Basic research with uncertain returns and long timelines doesn't fit the profit logic of private investment.",
        "blog_url": None,
    },
    "progress_collective_instrument": {
        "argument": "Technology makes government more effective",
        "logic": "Digital infrastructure, data systems, and communication tools allow governments to deliver services, enforce laws, and coordinate responses faster and more accurately.",
        "blog_url": None,
    },
    "collective_progress_tradeoff": {
        "argument": "Government regulation slows innovation",
        "logic": "Rules designed to prevent harm also prevent experiments. The same oversight that protects people from dangerous technology delays the development of beneficial technology.",
        "blog_url": None,
    },
    "progress_collective_spillover_neg": {
        "argument": "Technology concentrates power in ways that undermine democracy",
        "logic": "Surveillance tools, algorithmic control, and information systems developed for one purpose become instruments of government control when those in power choose to use them that way.",
        "blog_url": None,
    },
    "collective_progress_synergy": {
        "argument": "Public investment and private innovation build each other",
        "logic": "Government funds the risky early research. Private companies commercialize the results. The division of labor is stable because both sides get what they need from it.",
        "blog_url": None,
    },
    "progress_collective_blocking": {
        "argument": "Technology makes collective governance harder",
        "logic": "When information spreads faster than institutions can process it, when platforms operate across borders, and when algorithms shape public opinion, traditional governance loses its grip.",
        "blog_url": None,
    },
    # ── Pair 17: COLLECTIVE ↔ PRESERVATION (5) ──
    "collective_preservation_instrument": {
        "argument": "Government protection is what saves culture from market forces",
        "logic": "Languages, arts, and traditions that don't generate profit disappear without institutional support. Markets preserve what sells. Everything else needs collective investment.",
        "blog_url": None,
    },
    "preservation_collective_instrument": {
        "argument": "Shared cultural identity is what makes collective action possible",
        "logic": "Communities act together more effectively when they share history, values, and identity. Culture is not separate from governance — it is what governance is built on.",
        "blog_url": None,
    },
    "collective_preservation_tradeoff": {
        "argument": "Government cultural policy always involves choosing whose culture to protect",
        "logic": "Public funding for heritage means deciding what counts as heritage. That decision is never neutral — it reflects existing power and inevitably favors some groups over others.",
        "blog_url": None,
    },
    "collective_preservation_spillover_neg": {
        "argument": "Well-intentioned cultural policy can freeze traditions instead of keeping them alive",
        "logic": "When governments officially define and fund a cultural practice, they often fix it in one form. Living traditions adapt. Protected ones become museum pieces.",
        "blog_url": None,
    },
    "preservation_collective_causal_pos": {
        "argument": "Strong cultural identity holds communities together through change",
        "logic": "Societies facing rapid economic or technological disruption hold together better when people share a sense of who they are collectively. Culture provides the continuity institutions need.",
        "blog_url": None,
    },
    # ── Pair 18: COLLECTIVE ↔ FLOURISHING (5) ──
    "collective_flourishing_instrument": {
        "argument": "Public systems are what make human wellbeing possible at scale",
        "logic": "A child whose family cannot afford therapy still gets help if the school provides a counsellor. A worker who loses a job does not lose their home if unemployment insurance exists. Wellbeing without public systems is a privilege limited to those who can pay.",
        "blog_url": None,
    },
    "flourishing_collective_instrument": {
        "argument": "Human wellbeing is the ultimate purpose of collective organization",
        "logic": "A country with high GDP but epidemic loneliness and rising suicide rates has failed at the only thing governments exist to do. Institutions that cannot point to improved human lives have no justification for the power they hold.",
        "blog_url": None,
    },
    "collective_flourishing_tradeoff": {
        "argument": "Social obligations limit personal wellbeing",
        "logic": "Taxes, duties, regulations, and community expectations all ask individuals to give something up. The collective good and personal satisfaction are not always pointing the same direction.",
        "blog_url": None,
    },
    "collective_flourishing_causal_pos": {
        "argument": "Social connection is essential to human wellbeing",
        "logic": "Loneliness increases the risk of early death more than smoking 15 cigarettes a day. Community centres, public spaces, and shared institutions are not luxuries — they are the infrastructure that prevents people from dying alone.",
        "blog_url": None,
    },
    "flourishing_collective_blocking": {
        "argument": "A society of unhappy people cannot sustain effective collective life",
        "logic": "Mental illness, burnout, and social disconnection reduce people's capacity to contribute, cooperate, and participate. Neglecting individual wellbeing eventually undermines everything built collectively.",
        "blog_url": None,
    },
    # ── Pair 19: PROGRESS ↔ PRESERVATION (6) ──
    "progress_preservation_causal_neg": {
        "argument": "Innovation destroys traditions that cannot be recovered",
        "logic": "Once a language, craft, or way of life is gone, it is gone. Technological change moves faster than communities can adapt, and the losses are permanent.",
        "blog_url": None,
    },
    "progress_preservation_instrument_pos": {
        "argument": "Technology can preserve and revive cultural heritage",
        "logic": "Digital archives, language learning apps, and online communities allow traditions to survive and spread in ways that would have been impossible without modern tools.",
        "blog_url": None,
    },
    "preservation_progress_blocking": {
        "argument": "Attachment to tradition slows necessary change",
        "logic": "Practices defended because they are familiar rather than because they work prevent communities from adopting better solutions. Not all tradition is wisdom — some is just habit.",
        "blog_url": None,
    },
    "preservation_progress_informing": {
        "argument": "Traditional knowledge guides better innovation",
        "logic": "Indigenous land management, traditional medicine, and centuries-old farming techniques often contain insights that modern research eventually validates. Old knowledge and new methods work better together.",
        "blog_url": None,
    },
    "progress_preservation_tradeoff": {
        "argument": "Societies have to choose how fast to change",
        "logic": "Rapid technological adoption produces economic gains but disrupts the social fabric. Slower adoption preserves cohesion but risks falling behind. The pace of change is itself a choice with real consequences.",
        "blog_url": None,
    },
    "progress_preservation_transformation": {
        "argument": "What innovators call progress, traditionalists experience as loss",
        "logic": "Online shopping is convenience to one person and the death of their town's high street to another. The same change produces genuinely different realities for different people.",
        "blog_url": None,
    },
    # ── Pair 20: PROGRESS ↔ FLOURISHING (8) ──
    "progress_flourishing_causal_pos": {
        "argument": "Technology improves human wellbeing",
        "logic": "A diabetic today monitors blood sugar with a phone app that would have required a hospital visit 20 years ago. A grandparent video-calls grandchildren across continents daily. Technology did not just add convenience — it removed specific sources of suffering.",
        "blog_url": None,
    },
    "progress_flourishing_causal_neg": {
        "argument": "Technology harms human wellbeing",
        "logic": "Living faster, always online, and constantly compared to others makes people anxious, distracted, and less satisfied with their lives.",
        "blog_url": None,
    },
    "progress_flourishing_instrument": {
        "argument": "Technology is only valuable if it serves humans",
        "logic": "A faster phone or smarter algorithm means nothing on its own. The only question that matters is whether people's lives actually get better because of it.",
        "blog_url": None,
    },
    "progress_flourishing_spillover_neg": {
        "argument": "Technology designed for one purpose accidentally damages wellbeing",
        "logic": "Apps built to be useful end up being addictive. Tools built for convenience end up creating pressure. The harm was never the goal — it just came with it.",
        "blog_url": None,
    },
    "progress_flourishing_blocking": {
        "argument": "Obsession with efficiency leaves no room for human dignity",
        "logic": "When everything gets measured by speed and output, the things that make life meaningful — rest, craft, connection — get treated as waste.",
        "blog_url": None,
    },
    "progress_flourishing_transformation": {
        "argument": "What looks like progress from outside feels like loss from inside",
        "logic": "Automation replacing a job is a productivity gain on a spreadsheet and a loss of purpose for the person who held that job. Same event, completely different reality.",
        "blog_url": None,
    },
    "flourishing_progress_instrument": {
        "argument": "Happier people produce better innovation",
        "logic": "People who feel safe, motivated, and valued think more creatively and work more sustainably than people running on stress and fear.",
        "blog_url": None,
    },
    "progress_flourishing_feedback_neg": {
        "argument": "Technology and human distress feed each other",
        "logic": "Anxious people spend more time on their phones. More time on phones makes people more anxious. The cycle keeps going because distress is good for engagement numbers.",
        "blog_url": None,
    },
    # ── Pair 21: PRESERVATION ↔ FLOURISHING (6) ──
    "preservation_flourishing_causal_pos": {
        "argument": "Cultural roots give people a sense of meaning and belonging",
        "logic": "People who feel connected to a heritage, a community, and a history report stronger senses of identity and purpose than those who have been cut off from their cultural origins.",
        "blog_url": None,
    },
    "flourishing_preservation_instrument": {
        "argument": "Human wellbeing is the reason culture is worth preserving",
        "logic": "Traditions matter not as museum pieces but because they sustain the sense of identity, connection, and meaning that people need to live well. If a tradition no longer serves this, its value is questionable.",
        "blog_url": None,
    },
    "preservation_flourishing_tradeoff": {
        "argument": "Cultural expectations can limit individual flourishing",
        "logic": "Traditions that define gender roles, restrict personal choices, or demand conformity can prevent individuals from discovering and living according to their own values.",
        "blog_url": None,
    },
    "flourishing_preservation_causal_pos": {
        "argument": "Psychological wellbeing motivates people to keep traditions alive",
        "logic": "Practices that people find genuinely meaningful get passed on. Those that feel like obligations or performances do not. Inner life is what keeps culture living rather than just maintained.",
        "blog_url": None,
    },
    "preservation_flourishing_synergy": {
        "argument": "Cultural identity and personal wellbeing strengthen each other",
        "logic": "A teenager who knows their grandmother's language and stories navigates adolescent identity crises with something to stand on. That grounding makes them more confident, which makes them more willing to carry the culture forward. Roots feed resilience; resilience protects roots.",
        "blog_url": None,
    },
    "preservation_flourishing_blocking": {
        "argument": "Rigid cultural traditions can damage mental health",
        "logic": "When communities enforce conformity through shame, exclusion, or punishment, they harm the people who don't fit — particularly those whose identity conflicts with inherited norms.",
        "blog_url": None,
    },
}

# Cross-check against actual data
data_directions = set()
for e in essay_objects.values():
    data_directions.update(e['directions'])

missing = data_directions - set(DIRECTION_LOOKUP.keys())
if missing:
    print(f'⚠️  DIRECTION_LOOKUP missing {len(missing)} tags from data: {missing}')
else:
    print(f'✅ DIRECTION_LOOKUP — {len(DIRECTION_LOOKUP)} entries, covers all {len(data_directions)} tags in data')

✅ DIRECTION_LOOKUP — 43 entries, covers all 40 tags in data


---
## Cell 13 — Write `tier_unit_sequence` to Supabase

One row per step per tier. Maps sequence positions to unit_ids.

```sql
-- Reference: create this table in Supabase dashboard first
CREATE TABLE tier_unit_sequence (
    id           UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    batch_id     UUID NOT NULL,
    tier         TEXT NOT NULL,          -- 'tier_50' or 'tier_80'
    sequence_position INT NOT NULL,      -- 1-indexed
    unit_id      UUID NOT NULL,          -- = essay_id from P1
    UNIQUE(batch_id, tier, sequence_position)
);
```

In [13]:
# Build rows for tier_unit_sequence
tus_rows = []

for tier_name in ['tier_50', 'tier_80']:
    for step in results[tier_name]['steps']:
        tus_rows.append({
            'batch_id':          BATCH_ID,
            'tier':              tier_name,
            'sequence_position': step['step'],
            'unit_id':           step['model']['essay_id'],
        })

print(f'Rows to insert: {len(tus_rows)}')
print(f'  tier_50: {sum(1 for r in tus_rows if r["tier"] == "tier_50")}')
print(f'  tier_80: {sum(1 for r in tus_rows if r["tier"] == "tier_80")}')

# Insert into Supabase
resp = supabase.table('tier_unit_sequence').insert(tus_rows).execute()
print(f'\n✅ Wrote {len(resp.data)} rows to tier_unit_sequence')

Rows to insert: 20
  tier_50: 6
  tier_80: 14

✅ Wrote 20 rows to tier_unit_sequence


---
## Cell 14 — Write `qbank_unlocks` to Supabase

One row per Q Bank essay per tier. Includes `shared_directions` JSONB.

```sql
-- Reference: create this table in Supabase dashboard first
CREATE TABLE qbank_unlocks (
    id                            UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    batch_id                      UUID NOT NULL,
    tier                          TEXT NOT NULL,
    question_id                   UUID NOT NULL,         -- = essay_id for the Q Bank essay
    question_text                 TEXT NOT NULL,          -- denormalised from P1
    unlocked_by_sequence_position INT NOT NULL,
    shared_directions             JSONB NOT NULL DEFAULT '[]',
    UNIQUE(batch_id, tier, question_id)
);
```

In [14]:
# Build rows for qbank_unlocks
qbu_rows = []

for tier_name in ['tier_50', 'tier_80']:
    unlocks = results[tier_name]['qbank_unlocks']
    for essay_id, unlock_data in unlocks.items():
        essay = essay_objects[essay_id]
        qbu_rows.append({
            'batch_id':                      BATCH_ID,
            'tier':                          tier_name,
            'question_id':                   essay_id,
            'question_text':                 essay['question'],
            'unlocked_by_sequence_position': unlock_data['unlock_position'],
            'shared_directions':             unlock_data['shared_directions'],
        })

print(f'Rows to insert: {len(qbu_rows)}')
print(f'  tier_50: {sum(1 for r in qbu_rows if r["tier"] == "tier_50")}')
print(f'  tier_80: {sum(1 for r in qbu_rows if r["tier"] == "tier_80")}')

# Insert into Supabase
resp = supabase.table('qbank_unlocks').insert(qbu_rows).execute()
print(f'\n✅ Wrote {len(resp.data)} rows to qbank_unlocks')

Rows to insert: 108
  tier_50: 47
  tier_80: 61

✅ Wrote 108 rows to qbank_unlocks


---
## Cell 15 — Write `direction_ref` to Supabase

Lookup table: direction_id → argument text.

```sql
-- Reference: create this table in Supabase dashboard first
CREATE TABLE direction_ref (
    direction_id TEXT PRIMARY KEY,
    argument     TEXT NOT NULL
);
```

In [15]:
# Build rows for direction_ref (v3: includes logic + blog_url)
dref_rows = [
    {
        'direction_id': did,
        'argument':     entry['argument'],
        'logic':        entry['logic'],
        'blog_url':     entry['blog_url'],
    }
    for did, entry in DIRECTION_LOOKUP.items()
]

print(f'Rows to insert: {len(dref_rows)}')

# Upsert (in case direction_ref already exists from a previous run)
resp = supabase.table('direction_ref').upsert(dref_rows).execute()
print(f'\n✅ Wrote {len(resp.data)} rows to direction_ref')

Rows to insert: 43

✅ Wrote 43 rows to direction_ref


---
## Cell 16 — Build enriched snapshot for P3

Assembles the self-contained JSON file that Pipeline 3 reads.
Contains resolved essays with `similar_phrases` on lexical items,
`hint_sentences` on sentences, plus indices for MCQ distractor logic.

**Saved to Google Drive so P3 can load it in a separate Colab session.**

In [ ]:
import copy
from itertools import combinations


def resolve_and_strip(essay, lex_idx, phrase_lookup, hint_idx):
    """
    Deep copy an essay, resolve similar_phrases onto lexical items,
    attach hint_sentences onto sentences, then strip internal fields
    (lexical_id, embeddings).

    v5 fix: stamps essay_id onto each sentence so P3 can group by
    (essay_id, paragraph_type) after flattening into distractor pools.
    """
    e = copy.deepcopy(essay)
    eid = e.get('essay_id')  # v5 fix: capture essay_id for stamping
    for s in e['sentences']:
        s.pop('embedding', None)
        s['essay_id'] = eid  # v5 fix: stamp essay_id onto each sentence

        # Attach hint_sentences for L2F
        sid = s.get('sentence_id')
        s['hint_sentences'] = hint_idx.get(sid, []) if sid and hint_idx else []

        # Resolve similar_phrases on each lexical item
        cleaned_lex = []
        for li in s.get('lexical_items', []):
            if isinstance(li, dict):
                lid         = li.get('lexical_id')
                similar_ids = lex_idx.get(lid, []) if lid else []
                cleaned_lex.append({
                    'phrase':          li['phrase'],
                    'similar_phrases': [
                        phrase_lookup[sid]
                        for sid in similar_ids
                        if sid in phrase_lookup
                    ],
                })
        s['lexical_items'] = cleaned_lex
    return e


def build_rhetoric_index(essay_objects_dict, covered_ids):
    """Groups all sentence_ids by rhetoric_tag for fast P3 lookups."""
    index = defaultdict(list)
    for eid in covered_ids:
        for s in essay_objects_dict[eid]['sentences']:
            tag = s.get('rhetoric_tag')
            if tag:
                index[tag].append(s['sentence_id'])
    return dict(index)


def build_direction_pairs(essay_objects_dict, covered_ids):
    """All unique direction-tag pairs that co-occur in at least one essay."""
    pairs = set()
    for eid in covered_ids:
        dirs = essay_objects_dict[eid]['directions']
        if len(dirs) >= 2:
            for a, b in combinations(sorted(dirs), 2):
                pairs.add((a, b))
    return [list(pair) for pair in sorted(pairs)]


def assemble_tier(essay_objects_dict, result):
    """Build the complete tier sub-object for the P3 snapshot."""
    covered_ids = result['covered_ids']
    lex_idx     = result['lexical_similarity_index']
    hint_idx    = result['hint_sentences_index']

    # Build phrase lookup: lexical_id → phrase text
    phrase_lookup = {}
    for eid in covered_ids:
        for s in essay_objects_dict[eid]['sentences']:
            for li in s.get('lexical_items', []):
                if li.get('lexical_id') and li.get('phrase'):
                    phrase_lookup[li['lexical_id']] = li['phrase']

    # Sequence (ordered greedy steps)
    sequence = [
        {
            'step':           s['step'],
            'new_directions': s['new_directions'],
            'model': resolve_and_strip(s['model'], lex_idx, phrase_lookup, hint_idx) if s['model'] else None,
        }
        for s in result['steps']
    ]

    # Pool (all essays in tier — P3's distractor database)
    pool = [
        resolve_and_strip(essay_objects_dict[eid], lex_idx, phrase_lookup, hint_idx)
        for eid in covered_ids
    ]

    # Indices
    similarity_index = result['similarity_index']
    rhetoric_index   = build_rhetoric_index(essay_objects_dict, covered_ids)
    direction_pairs  = build_direction_pairs(essay_objects_dict, covered_ids)
    qbank_unlocks    = result['qbank_unlocks']

    return {
        'sequence':         sequence,
        'pool':             pool,
        'similarity_index': similarity_index,
        'rhetoric_index':   rhetoric_index,
        'direction_pairs':  direction_pairs,
        'qbank_unlocks':    qbank_unlocks,
    }


# Assemble full snapshot
snapshot = {
    'batch_id':     BATCH_ID,
    'generated_at': GENERATED_AT,
    'tier_50':      assemble_tier(essay_objects, results['tier_50']),
    'tier_80':      assemble_tier(essay_objects, results['tier_80']),
}

# Print summary
print('Snapshot assembled:')
for label in ['tier_50', 'tier_80']:
    t = snapshot[label]
    print(f'  {label}:')
    print(f'    sequence steps    : {len(t["sequence"])}')
    print(f'    pool essays       : {len(t["pool"])}')
    print(f'    similarity_index  : {len(t["similarity_index"])} sentences')
    print(f'    rhetoric_index    : {len(t["rhetoric_index"])} tags')
    print(f'    direction_pairs   : {len(t["direction_pairs"])} pairs')
    print(f'    qbank_unlocks     : {len(t["qbank_unlocks"])} Q Bank essays')

print(f'\n  batch_id : {BATCH_ID}')
print('\n✅ Snapshot ready')

---
## Cell 17 — Save snapshot + upload to Google Drive

Saves `p2_enriched_snapshot.json` locally, then uploads to the shared
Google Drive folder where P3 will download it.

In [17]:
import json
import os

OUTPUT_FILE = 'p2_enriched_snapshot.json'

# Save to Colab disk
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(snapshot, f, indent=2, ensure_ascii=False)

file_size_mb = os.path.getsize(OUTPUT_FILE) / 1024 / 1024
print(f'✅ Saved → {OUTPUT_FILE}')
print(f'   File size : {file_size_mb:.2f} MB')
print(f'   batch_id  : {BATCH_ID}')

✅ Saved → p2_enriched_snapshot.json
   File size : 8.94 MB
   batch_id  : 00b60e61-22d9-4513-8d08-a8f7c09e6c79


In [18]:
# Upload to Google Drive
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

auth.authenticate_user()
service = build('drive', 'v3')

FOLDER_ID   = '1eOMwvWuJw_BsOaQXcgTKVLx2ykK77UXE'  # same folder as before
UPLOAD_FILE = 'p2_enriched_snapshot.json'
UPLOAD_NAME = 'p2_enriched_snapshot.json'

# Check if file already exists (to replace it)
existing = service.files().list(
    q=f"name='{UPLOAD_NAME}' and '{FOLDER_ID}' in parents and trashed=false",
    fields='files(id, name)'
).execute().get('files', [])

media = MediaFileUpload(UPLOAD_FILE, mimetype='application/json', resumable=False)

if existing:
    file_id = existing[0]['id']
    service.files().update(fileId=file_id, media_body=media).execute()
    print(f'♻️  Replaced existing {UPLOAD_NAME} in Drive.')
else:
    meta   = {'name': UPLOAD_NAME, 'parents': [FOLDER_ID]}
    result = service.files().create(body=meta, media_body=media, fields='id').execute()
    print(f'✅ Uploaded {UPLOAD_NAME} to Drive.')

print(f'   Folder: https://drive.google.com/drive/folders/{FOLDER_ID}')

✅ Uploaded p2_enriched_snapshot.json to Drive.
   Folder: https://drive.google.com/drive/folders/1eOMwvWuJw_BsOaQXcgTKVLx2ykK77UXE


---
## Cell 18 — Verification checklist

Validates all Supabase writes + snapshot integrity.

In [19]:
passes   = []
failures = []

def check(label, condition, detail=''):
    if condition:
        passes.append(f'  ✅  {label}')
    else:
        failures.append(f'  ❌  {label}' + (f'\n       → {detail}' if detail else ''))


# ── Supabase table checks ─────────────────────────────────────────────────

# tier_unit_sequence
tus_check = supabase.table('tier_unit_sequence').select('*').eq('batch_id', BATCH_ID).execute()
check('tier_unit_sequence has rows', len(tus_check.data) > 0, f'Got {len(tus_check.data)}')
check('tier_unit_sequence count matches', len(tus_check.data) == len(tus_rows), f'Expected {len(tus_rows)}, got {len(tus_check.data)}')

# qbank_unlocks
qbu_check = supabase.table('qbank_unlocks').select('*').eq('batch_id', BATCH_ID).execute()
check('qbank_unlocks has rows', len(qbu_check.data) > 0, f'Got {len(qbu_check.data)}')
check('qbank_unlocks count matches', len(qbu_check.data) == len(qbu_rows), f'Expected {len(qbu_rows)}, got {len(qbu_check.data)}')

# Check shared_directions is non-empty on at least some rows
shared_nonempty = sum(1 for r in qbu_check.data if r.get('shared_directions'))
check('qbank_unlocks has shared_directions', shared_nonempty > 0, f'{shared_nonempty}/{len(qbu_check.data)} have data')

# direction_ref
dref_check = supabase.table('direction_ref').select('*').execute()
check('direction_ref has rows', len(dref_check.data) >= len(DIRECTION_LOOKUP), f'Got {len(dref_check.data)}')

# ── Snapshot integrity checks ──────────────────────────────────────────────

check('snapshot batch_id matches', snapshot['batch_id'] == BATCH_ID)

for tier_label in ['tier_50', 'tier_80']:
    t   = snapshot[tier_label]
    seq = t['sequence']
    prefix = tier_label

    check(f'{prefix} — sequence non-empty', len(seq) > 0)

    # All model essays have no embedding vectors
    found_emb = 0
    for step in seq:
        if step.get('model'):
            for s in step['model']['sentences']:
                if 'embedding' in s:
                    found_emb += 1
    check(f'{prefix} — no embeddings in sequence', found_emb == 0, f'Found {found_emb}')

    # All lexical items have similar_phrases key
    missing_sp = 0
    for essay in t['pool']:
        for s in essay['sentences']:
            for li in s.get('lexical_items', []):
                if 'similar_phrases' not in li:
                    missing_sp += 1
    check(f'{prefix} — all lexical items have similar_phrases', missing_sp == 0, f'{missing_sp} missing')

    # All sentences have hint_sentences key
    missing_hs = 0
    for essay in t['pool']:
        for s in essay['sentences']:
            if 'hint_sentences' not in s:
                missing_hs += 1
    check(f'{prefix} — all sentences have hint_sentences', missing_hs == 0, f'{missing_hs} missing')

    # No lexical_id UUIDs in pool
    found_lid = 0
    for essay in t['pool']:
        for s in essay['sentences']:
            for li in s.get('lexical_items', []):
                if 'lexical_id' in li:
                    found_lid += 1
    check(f'{prefix} — no lexical_id in pool', found_lid == 0, f'Found {found_lid}')

# ── Cross-check: Supabase rows match snapshot ──────────────────────────────

# Every unit_id in tier_unit_sequence should have a model essay in the snapshot
tus_unit_ids = {r['unit_id'] for r in tus_check.data}
snap_model_ids = set()
for tier_label in ['tier_50', 'tier_80']:
    for step in snapshot[tier_label]['sequence']:
        if step.get('model'):
            snap_model_ids.add(step['model']['essay_id'])
missing_units = tus_unit_ids - snap_model_ids
check('All tier_unit_sequence unit_ids in snapshot', len(missing_units) == 0, f'Missing: {missing_units}')


# ── Print results ──────────────────────────────────────────────────────────
print('=' * 60)
print('PIPELINE 2 v3 — VERIFICATION')
print('=' * 60)
for msg in passes:
    print(msg)
for msg in failures:
    print(msg)
print('=' * 60)
print(f'  {len(passes)} passed   {len(failures)} failed')

if not failures:
    print('\n🎉 Pipeline 2 complete — all checks passed.')
    print(f'   batch_id: {BATCH_ID}')
    print(f'   Next: Run Pipeline 3 with this batch_id.')
else:
    print('\n🚨 Fix failures above before running Pipeline 3.')

PIPELINE 2 v3 — VERIFICATION
  ✅  tier_unit_sequence has rows
  ✅  tier_unit_sequence count matches
  ✅  qbank_unlocks has rows
  ✅  qbank_unlocks count matches
  ✅  qbank_unlocks has shared_directions
  ✅  direction_ref has rows
  ✅  snapshot batch_id matches
  ✅  tier_50 — sequence non-empty
  ✅  tier_50 — no embeddings in sequence
  ✅  tier_50 — all lexical items have similar_phrases
  ✅  tier_50 — all sentences have hint_sentences
  ✅  tier_50 — no lexical_id in pool
  ✅  tier_80 — sequence non-empty
  ✅  tier_80 — no embeddings in sequence
  ✅  tier_80 — all lexical items have similar_phrases
  ✅  tier_80 — all sentences have hint_sentences
  ✅  tier_80 — no lexical_id in pool
  ✅  All tier_unit_sequence unit_ids in snapshot
  18 passed   0 failed

🎉 Pipeline 2 complete — all checks passed.
   batch_id: 00b60e61-22d9-4513-8d08-a8f7c09e6c79
   Next: Run Pipeline 3 with this batch_id.
